# Atlas v1 — first real adaptive DP622 production campaign

Atlas reconstructs an **active-like** DP622–Aβ/Zn model and runs the sole production authority, `atlas adaptive-run`, through mechanism-aware generation, genuine ThermoMPNN/ThermoMPNN-D inference, candidate-specific structures, adversarial review, and at most five experimentally untested wet-lab hypotheses. Replicated explicit-solvent MD is excluded from candidate discrimination after failed reference validation.

## How to run this notebook

1. Open this notebook in Google Colab.
2. Select **Runtime → Change runtime type → T4 GPU**.
3. Run the configuration and hardware-check cells.
4. Confirm the full preflight reports `passed: true`.
5. Run the remaining cells in order.
6. Do not rerun ThermoMPNN or ThermoMPNN-D cells unnecessarily; valid checkpoints are reused automatically.
7. Find the completion audit, finalist dossiers, and machine-readable outputs in the printed adaptive run directory. The last cell downloads a ZIP.

## Configuration — review this before running

In [ ]:
ATLAS_REPO_URL = 'https://github.com/Noelduval/atlas-therapeutic-optimization.git'
ATLAS_REF = 'aa5f61d8a5fbaffb50e1c2fef3b6b3cc275f707c'
THERMOMPNN_REV = '2b04fd370e399911b1fa5848112cc9013f084110'
THERMOMPNN_D_REV = 'df9a75aaddb674a7c4c193005031fc0536d325fb'
USE_GOOGLE_DRIVE = True  # Recommended: checkpoints survive runtime restarts.
print('Configured Atlas ref:', ATLAS_REF)

## Stage 1 — fast T4 hardware check

In [ ]:
import subprocess
class StageExecutionError(RuntimeError):
    pass

def run_bootstrap_command(stage_name, command, cwd=None):
    import os
    import shlex
    import time
    working_directory = os.path.abspath(cwd or os.getcwd())
    exact_command = shlex.join(str(part) for part in command)
    started = time.perf_counter()
    print(f'\n=== START: {stage_name} ===')
    print('Interpreter:', command[0] if command else '<empty command>')
    print('Working directory:', working_directory)
    print('Exact command:', exact_command, flush=True)
    environment = os.environ.copy()
    environment['MPLBACKEND'] = 'Agg'
    completed = subprocess.run(
        [str(part) for part in command],
        cwd=working_directory,
        text=True,
        capture_output=True,
        check=False,
        env=environment,
    )
    if completed.returncode:
        print('\n=== NOTEBOOK SETUP STAGE FAILED ===')
        print('Stage:', stage_name)
        print('Exit status:', completed.returncode)
        print('Working directory:', working_directory)
        print('Interpreter:', command[0] if command else '<empty command>')
        print('Exact command:', exact_command)
        print('--- complete stdout ---')
        print(completed.stdout or '<empty>')
        print('--- complete stderr ---')
        print(completed.stderr or '<empty>')
        print('Suggested next action: fix the first setup error above, then rerun this cell.')
        print(f'Elapsed seconds: {time.perf_counter() - started:.2f}')
        raise StageExecutionError(f'{stage_name} failed with exit status {completed.returncode}')
    if completed.stdout:
        print(completed.stdout, end='' if completed.stdout.endswith('\n') else '\n')
    if completed.stderr:
        print(completed.stderr, end='' if completed.stderr.endswith('\n') else '\n')
    print(f'PASS: {stage_name}')
    print(f'Elapsed seconds: {time.perf_counter() - started:.2f}')
    return completed

print('Checking the GPU before installing or loading scientific models...')
gpu = run_bootstrap_command(
    'T4 hardware check',
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free,memory.used', '--format=csv,noheader'],
)
print(gpu.stdout.strip())
if 'T4' not in gpu.stdout:
    raise RuntimeError('This reviewer path is configured for a Tesla T4. Select a T4 GPU runtime and reconnect.')
print('Host Python hardware orchestrator only; scientific CUDA is checked after the pinned environment is installed.')

## Stage 2 — exact repository setup and persistent checkpoint directory

In [ ]:
import hashlib
import os
from pathlib import Path
import subprocess
import sys

ATLAS_DIR = Path('/content/Atlas')
if not (ATLAS_DIR / '.git').exists():
    run_bootstrap_command('Clone Atlas', ['git', 'clone', '--no-checkout', ATLAS_REPO_URL, str(ATLAS_DIR)])
run_bootstrap_command('Fetch configured Atlas ref', ['git', '-C', str(ATLAS_DIR), 'fetch', 'origin', ATLAS_REF])
run_bootstrap_command('Check out configured Atlas ref', ['git', '-C', str(ATLAS_DIR), 'checkout', '--detach', 'FETCH_HEAD'])
ATLAS_SHA = run_bootstrap_command(
    'Resolve Atlas commit',
    ['git', '-C', str(ATLAS_DIR), 'rev-parse', 'HEAD'],
).stdout.strip()
print('Resolved Atlas commit:', ATLAS_SHA)

def build_scientific_environment_commands(atlas_dir, environment_dir, host_python):
    atlas_dir = Path(atlas_dir)
    environment_dir = Path(environment_dir)
    uv = [str(host_python), '-m', 'uv']
    scientific_python = str(environment_dir / 'bin/python')
    return [
        [str(host_python), '-m', 'pip', 'install', '--quiet', 'uv==0.8.13'],
        [*uv, 'venv', '--python', '3.10', '--managed-python', str(environment_dir)],
        [
            *uv, 'pip', 'install', '--python', scientific_python,
            '--index-url', 'https://download.pytorch.org/whl/cu118',
            'torch==2.5.1', 'torchvision==0.20.1', 'torchaudio==2.5.1',
        ],
        [
            *uv, 'pip', 'install', '--python', scientific_python,
            f'{atlas_dir}[dynamics]',
            'biopython==1.85', 'matplotlib==3.9.2', 'numpy==2.1.3',
            'pandas==2.2.3', 'typer==0.16.1', 'openmm==8.2.0',
            'openmm-cuda-12==8.2.0',
            'omegaconf==2.3.0', 'wandb==0.18.7',
            'pytorch-lightning==2.4.0', 'scipy==1.14.1',
            'scikit-learn==1.5.2', 'joblib==1.4.2',
            'tqdm==4.67.1', 'torchmetrics==1.6.0',
        ],
        [
            *uv, 'pip', 'install', '--python', scientific_python,
            '--reinstall', '--no-deps', str(atlas_dir),
        ],
    ]

SCIENTIFIC_ENV = Path('/content/atlas-science')
SCIENTIFIC_PYTHON = SCIENTIFIC_ENV / 'bin/python'
environment_commands = build_scientific_environment_commands(ATLAS_DIR, SCIENTIFIC_ENV, sys.executable)
print('Creating the pinned Python 3.10 scientific runtime...')
run_bootstrap_command('Install pinned uv orchestrator', environment_commands[0])
if not SCIENTIFIC_PYTHON.is_file():
    run_bootstrap_command('Create managed CPython 3.10 environment', environment_commands[1])
run_bootstrap_command('Install pinned PyTorch CUDA 11.8 stack', environment_commands[2])
run_bootstrap_command('Install pinned Atlas scientific dependencies', environment_commands[3])
run_bootstrap_command('Install the resolved Atlas checkout exactly', environment_commands[4])

EXTERNAL = ATLAS_DIR / '.external'
EXTERNAL.mkdir(exist_ok=True)
model_repositories = [
    ('https://github.com/Kuhlman-Lab/ThermoMPNN.git', EXTERNAL / 'ThermoMPNN', THERMOMPNN_REV),
    ('https://github.com/Kuhlman-Lab/ThermoMPNN-D.git', EXTERNAL / 'ThermoMPNN-D', THERMOMPNN_D_REV),
]
for url, path, revision in model_repositories:
    if not (path / '.git').exists():
        run_bootstrap_command(f'Clone {path.name}', ['git', 'clone', '--no-checkout', url, str(path)])
    run_bootstrap_command(f'Fetch pinned {path.name}', ['git', '-C', str(path), 'fetch', 'origin', revision])
    run_bootstrap_command(f'Check out pinned {path.name}', ['git', '-C', str(path), 'checkout', '--detach', 'FETCH_HEAD'])
    actual = run_bootstrap_command(
        f'Resolve {path.name} commit',
        ['git', '-C', str(path), 'rev-parse', 'HEAD'],
    ).stdout.strip()
    if actual != revision:
        raise RuntimeError(f'Wrong checkout for {path.name}: {actual}')
    print(f'{path.name} commit: {actual}')

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = Path('/content/drive/MyDrive/Atlas/checkpoints')
else:
    OUTPUT_ROOT = ATLAS_DIR / 'outputs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
INPUT_STRUCTURE = ATLAS_DIR / 'data/23WN.cif'
input_sha = hashlib.sha256(INPUT_STRUCTURE.read_bytes()).hexdigest()
RUN_ID = f'atlas-t4-{ATLAS_SHA[:12]}-{input_sha[:8]}'
RUN_DIR = OUTPUT_ROOT / RUN_ID
os.chdir(ATLAS_DIR)
print('Checkpoint run directory:', RUN_DIR)

In [ ]:
import hashlib
import platform
from pathlib import Path
import sys

print('Host Python:', sys.executable, platform.python_version())
provenance_script = f'''
import hashlib
from pathlib import Path
import sys
import torch
import openmm
import atlas
if sys.version_info[:2] != (3, 10):
    raise RuntimeError(f'Expected scientific Python 3.10, found {{sys.version}}')
print('Atlas commit: {ATLAS_SHA}')
print('Atlas package:', Path(atlas.__file__).resolve())
print('Scientific Python executable:', sys.executable)
print('Scientific Python version:', sys.version.replace('\\n', ' '))
print('PyTorch version:', torch.__version__)
print('CUDA runtime:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Pinned scientific PyTorch cannot see CUDA')
print('GPU:', torch.cuda.get_device_name(0))
openmm_cuda = openmm.Platform.getPlatformByName('CUDA')
print('OpenMM version:', openmm.version.full_version)
print('OpenMM CUDA platform:', openmm_cuda.getName())
print('ThermoMPNN commit: {THERMOMPNN_REV}')
print('ThermoMPNN-D commit: {THERMOMPNN_D_REV}')
print('23WN SHA256: {input_sha}')
print('Run ID: {RUN_ID}')
print('Checkpoint directory: {RUN_DIR}')
'''
run_bootstrap_command('Scientific runtime provenance', [str(SCIENTIFIC_PYTHON), '-c', provenance_script], ATLAS_DIR)

## Configure documented upstream runtime paths

In [ ]:
configure_script = f'''
from atlas.colab import configure_upstream_runtime_paths
for path in configure_upstream_runtime_paths({str(EXTERNAL / 'ThermoMPNN')!r}, {str(EXTERNAL / 'ThermoMPNN-D')!r}):
    print('Configured upstream checkout path:', path)
'''
run_bootstrap_command('Configure pinned upstream runtime paths', [str(SCIENTIFIC_PYTHON), '-c', configure_script], ATLAS_DIR)

## Full preflight — stop here unless every check passes

In [ ]:
import json
preflight_path = OUTPUT_ROOT / f'preflight-{ATLAS_SHA[:12]}.json'
preflight_command = [
    str(SCIENTIFIC_PYTHON), '-m', 'atlas', 'preflight',
    '--input', str(INPUT_STRUCTURE),
    '--atlas-repo', str(ATLAS_DIR),
    '--thermompnn-repo', str(EXTERNAL / 'ThermoMPNN'),
    '--thermompnn-d-repo', str(EXTERNAL / 'ThermoMPNN-D'),
    '--output-json', str(preflight_path),
]
print('Running the complete lightweight preflight...')
run_bootstrap_command(
    'Complete Atlas environment preflight',
    preflight_command,
    ATLAS_DIR,
)
preflight = json.loads(preflight_path.read_text())
print(json.dumps(preflight, indent=2))
if not preflight['passed']:
    raise RuntimeError('Preflight did not pass. Do not start model inference.')

## Runtime readiness — final cheap checks before Stage 3

In [ ]:
print('Checking Atlas imports, CLI entrypoint, repository/model layout, and checkpoint writability...')
readiness_script = f'''
import json
from atlas.colab import validate_colab_readiness
report = validate_colab_readiness(
    python_executable={str(SCIENTIFIC_PYTHON)!r},
    atlas_repo={str(ATLAS_DIR)!r},
    input_structure={str(INPUT_STRUCTURE)!r},
    thermompnn_repo={str(EXTERNAL / 'ThermoMPNN')!r},
    thermompnn_d_repo={str(EXTERNAL / 'ThermoMPNN-D')!r},
    output_root={str(OUTPUT_ROOT)!r},
    run_dir={str(RUN_DIR)!r},
)
print(json.dumps(report, indent=2))
'''
readiness = run_bootstrap_command('Scientific runtime readiness', [str(SCIENTIFIC_PYTHON), '-c', readiness_script], ATLAS_DIR)
runtime_readiness = json.loads(readiness.stdout)

# Adaptive production authority

This section runs the approved mechanism-aware search. The historical benchmark remains a frozen methodological characterization, not a prospective gate. No legacy generator, ranker, validation-gated workflow, or replicated-MD stage participates in finalist selection.

In [ ]:
def build_adaptive_stage_command(*, python_executable, input_structure, output_root, atlas_repo, thermompnn_repo, thermompnn_d_repo, run_id, candidate_budget, broad_target, structure_target, adversarial_target, portfolio_target, seed, stop_after=None, resume=False):
    command = [
        str(python_executable), '-m', 'atlas', 'adaptive-run',
        '--input', str(input_structure), '--output-root', str(output_root),
        '--atlas-repo', str(atlas_repo), '--thermompnn-repo', str(thermompnn_repo),
        '--thermompnn-d-repo', str(thermompnn_d_repo), '--run-id', run_id,
        '--candidate-budget', str(candidate_budget), '--broad-target', str(broad_target),
        '--structure-target', str(structure_target),
        '--adversarial-target', str(adversarial_target), '--portfolio-target', str(portfolio_target),
        '--seed', str(seed),
    ]
    if resume:
        command.append('--resume')
    if stop_after:
        command.extend(['--stop-after', stop_after])
    return command

ADAPTIVE_RUN_ID = f'atlas-adaptive-t4-{ATLAS_SHA[:12]}-{input_sha[:8]}'
ADAPTIVE_RUN_DIR = OUTPUT_ROOT / ADAPTIVE_RUN_ID
ADAPTIVE_STAGES = ('setup', 'round1', 'round2', 'round3', 'broad', 'structure', 'repair', 'adversarial')

def run_adaptive_stage(label, stop_after=None):
    command = build_adaptive_stage_command(
        python_executable=str(SCIENTIFIC_PYTHON),
        input_structure=INPUT_STRUCTURE,
        output_root=OUTPUT_ROOT,
        atlas_repo=ATLAS_DIR,
        thermompnn_repo=EXTERNAL / 'ThermoMPNN',
        thermompnn_d_repo=EXTERNAL / 'ThermoMPNN-D',
        run_id=ADAPTIVE_RUN_ID,
        candidate_budget=5000,
        broad_target=500,
        structure_target=100,
        adversarial_target=10,
        portfolio_target=5,
        seed=622,
        stop_after=stop_after,
        resume=(ADAPTIVE_RUN_DIR / 'run_context.json').exists(),
    )
    return run_bootstrap_command(label, command, ATLAS_DIR)

print('Prospective checkpoint directory:', ADAPTIVE_RUN_DIR)
print('Exact Atlas SHA:', ATLAS_SHA)
print('Stages:', ADAPTIVE_STAGES)

## Execute or resume the adaptive funnel

Every command reopens only an exact-context checkpoint. Completed expensive stages are verified and skipped. The final invocation runs the adversarial portfolio, completion audit, and reporting layer.

In [ ]:
for stage in ADAPTIVE_STAGES:
    run_adaptive_stage(f'Adaptive Atlas through {stage}', stage)
run_adaptive_stage('Finalize adaptive completion audit and reports')
execution = json.loads((ADAPTIVE_RUN_DIR / 'execution_status.json').read_text())
print(json.dumps(execution, indent=2))
if execution['status'] != 'completed':
    raise RuntimeError(f"Adaptive completion audit did not pass: {execution['completion_blockers']}")

## Bounded late-stage evidence and finalist dossiers

This consumes only the completed adversarial checkpoint. It does not rerun generation, ThermoMPNN/ThermoMPNN-D, candidate structures, or dynamics.

In [ ]:
late_stage_command = [
    str(SCIENTIFIC_PYTHON), '-m', 'atlas', 'late-stage',
    '--run-dir', str(ADAPTIVE_RUN_DIR), '--input', str(INPUT_STRUCTURE),
    '--seed', '622', '--resume',
]
run_bootstrap_command('Late-stage evidence and finalist dossiers', late_stage_command, ATLAS_DIR)
late_stage = json.loads((ADAPTIVE_RUN_DIR / 'late_stage/completion.json').read_text())
print(json.dumps(late_stage, indent=2))
if late_stage['status'] != 'completed':
    raise RuntimeError('Late-stage evidence did not complete')

## Inspect and export the prospective result

In [ ]:
import shutil
import pandas as pd
from IPython.display import display, Markdown, Image
display(pd.read_csv(ADAPTIVE_RUN_DIR / 'candidates.csv').head(20))
display(Markdown((ADAPTIVE_RUN_DIR / 'reports/atlas_final_design_report.md').read_text()))
for figure in sorted((ADAPTIVE_RUN_DIR / 'figures').glob('*.png')):
    display(Image(filename=str(figure)))
archive = shutil.make_archive(str(ADAPTIVE_RUN_DIR), 'zip', root_dir=ADAPTIVE_RUN_DIR.parent, base_dir=ADAPTIVE_RUN_DIR.name)
print('Adaptive report:', ADAPTIVE_RUN_DIR / 'reports/atlas_final_design_report.md')
print('Completion audit:', ADAPTIVE_RUN_DIR / 'completion_audit.json')
print('Reproducibility manifest:', ADAPTIVE_RUN_DIR / 'reproducibility_manifest.json')
print('Checkpointed ZIP:', archive)
from google.colab import files
files.download(archive)